A Wrapper Method is a feature selection technique in which a machine learning algorithm is used to evaluate different subsets of features and select the combination that gives the best model performance.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd

df = pd.read_csv("/content/drive/MyDrive/Teen_Mental_Health_Dataset.csv")

In [ ]:
from sklearn.preprocessing import LabelEncoder

X = df.drop("depression_label", axis=1)
y = df["depression_label"]

categorical = [
    "gender",
    "platform_usage",
    "social_interaction_level"
]

le = LabelEncoder()

for col in categorical:
    X[col] = le.fit_transform(X[col])

Forward Selection starts with zero features. It adds one feature at a time. After each addition, it checks whether the model accuracy improves. If yes, keep the feature. Otherwise, discard it.

No Features -> Age -> Age + Stress -> Age + Stress + Anxiety -> Age + Stress + Anxiety + Sleep -> Stop

In [ ]:
from sklearn.linear_model import LogisticRegression
from mlxtend.feature_selection import SequentialFeatureSelector

model = LogisticRegression(max_iter=1000)

forward = SequentialFeatureSelector(
    model,
    k_features=5,
    forward=True,
    floating=False,
    scoring="accuracy",
    cv=5
)

forward.fit(X, y)

SequentialFeatureSelector(estimator=LogisticRegression(max_iter=1000),
                          k_features=(5, 5), scoring='accuracy')

In [ ]:
selected = list(forward.k_feature_names_)

print(selected)
print(forward.k_score_)

['age', 'gender', 'daily_social_media_hours', 'platform_usage', 'screen_time_before_sleep']
0.9741666666666667


In [ ]:
from sklearn.tree import DecisionTreeClassifier
from mlxtend.feature_selection import SequentialFeatureSelector

model1 = DecisionTreeClassifier(random_state=42)

forward1 = SequentialFeatureSelector(
    model,
    k_features=5,
    forward=True,
    scoring='accuracy',
    cv=5
)

forward1.fit(X, y)

print(forward.k_feature_names_)
print("Accuracy:", forward.k_score_)

('age', 'gender', 'platform_usage', 'social_interaction_level', 'stress_level')
Accuracy: 0.9574999999999999


Backward Elimination starts with all features. Then removes one feature at a time. If removing a feature does not reduce performance, it is permanently removed.

All Features -> Remove Gender -> Remove Platform -> Remove Age -> Best Features Remain

In [ ]:
backward = SequentialFeatureSelector(

    model,

    k_features=5,

    forward=False,

    floating=False,

    scoring="accuracy",

    cv=5

)

backward.fit(X, y)

SequentialFeatureSelector(estimator=LogisticRegression(max_iter=1000),
                          forward=False, k_features=(5, 5), scoring='accuracy')

In [ ]:
print(backward.k_feature_names_)
print(backward.k_score_)

('daily_social_media_hours', 'sleep_hours', 'screen_time_before_sleep', 'stress_level', 'anxiety_level')
0.9850000000000001


In [ ]:
print("Forward :", forward.k_feature_names_)

print("Backward :", backward.k_feature_names_)

Forward : ('age', 'gender', 'daily_social_media_hours', 'platform_usage', 'screen_time_before_sleep')
Backward : ('daily_social_media_hours', 'sleep_hours', 'screen_time_before_sleep', 'stress_level', 'anxiety_level')



This is probably the most popular wrapper method. It starts with all features. It trains the model. Ranks all features. Removes the least important feature. Repeats until the required number of features remain.

12 Features ->Train Model -> Remove Least Important -> 11 Features -> Train Again -> Remove Least Important -> Continue -> 5 Features

In [ ]:
from sklearn.feature_selection import RFE

model = LogisticRegression(max_iter=1000)

selector = RFE(

    estimator=model,

    n_features_to_select=5

)

selector.fit(X, y)

RFE(estimator=LogisticRegression(max_iter=1000), n_features_to_select=5)

In [ ]:
selected = X.columns[selector.support_]

print(selected)

Index(['gender', 'daily_social_media_hours', 'sleep_hours', 'stress_level',
       'anxiety_level'],
      dtype='object')


In [ ]:
ranking = pd.DataFrame({

    "Feature": X.columns,

    "Rank": selector.ranking_

})

print(ranking.sort_values("Rank"))

                     Feature  Rank
1                     gender     1
2   daily_social_media_hours     1
4                sleep_hours     1
9               stress_level     1
10             anxiety_level     1
6       academic_performance     2
7          physical_activity     3
5   screen_time_before_sleep     4
0                        age     5
8   social_interaction_level     6
3             platform_usage     7
11           addiction_level     8


RFECV (Recursive Feature Elimination with Cross Validation)

This is the smart version of RFE. Instead of asking How many features should I keep? RFECV automatically decides.

In [ ]:
from sklearn.feature_selection import RFECV

rfecv = RFECV(

    estimator=LogisticRegression(max_iter=1000),

    step=1,

    cv=5,

    scoring="accuracy"

)

rfecv.fit(X, y)

RFECV(cv=5, estimator=LogisticRegression(max_iter=1000), scoring='accuracy')

In [ ]:
print("Optimal Features")

print(rfecv.n_features_)

Optimal Features
7


In [ ]:
selected = X.columns[rfecv.support_]

print(selected)

Index(['gender', 'daily_social_media_hours', 'sleep_hours',
       'academic_performance', 'physical_activity', 'stress_level',
       'anxiety_level'],
      dtype='object')


In [ ]:
result = pd.DataFrame({

    "Feature": X.columns,

    "Selected": rfecv.support_,

    "Ranking": rfecv.ranking_

})

print(result)

                     Feature  Selected  Ranking
0                        age     False        3
1                     gender      True        1
2   daily_social_media_hours      True        1
3             platform_usage     False        5
4                sleep_hours      True        1
5   screen_time_before_sleep     False        2
6       academic_performance      True        1
7          physical_activity      True        1
8   social_interaction_level     False        4
9               stress_level      True        1
10             anxiety_level      True        1
11           addiction_level     False        6


In [ ]:
print("Forward Selection")
print(list(forward.k_feature_names_))

print()

print("Backward Elimination")
print(list(backward.k_feature_names_))

print()

print("RFE")
print(list(X.columns[selector.support_]))

print()

print("RFECV")
print(list(X.columns[rfecv.support_]))

Forward Selection
['age', 'gender', 'daily_social_media_hours', 'platform_usage', 'screen_time_before_sleep']

Backward Elimination
['daily_social_media_hours', 'sleep_hours', 'screen_time_before_sleep', 'stress_level', 'anxiety_level']

RFE
['gender', 'daily_social_media_hours', 'sleep_hours', 'stress_level', 'anxiety_level']

RFECV
['gender', 'daily_social_media_hours', 'sleep_hours', 'academic_performance', 'physical_activity', 'stress_level', 'anxiety_level']
